In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [98]:
from sklearn.metrics import mutual_info_score

In [99]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [101]:
df[['annual_income','lead_score']] = df[['annual_income','lead_score']].fillna(0)

In [102]:
df[['lead_source','industry','employment_status','location']] = df[['lead_source','industry','employment_status','location']].fillna('NA')

In [103]:
df.sample(10)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
1123,social_media,finance,NA,south_america,71064.0,2,5,0.55,1
1388,social_media,finance,self_employed,europe,74908.0,1,2,0.30,0
1418,organic_search,technology,employed,north_america,90984.0,3,7,0.74,1
4088,referral,education,self_employed,south_america,30572.0,3,7,0.76,1
4707,organic_search,NA,employed,asia,39792.0,3,7,0.70,1
3342,organic_search,healthcare,student,europe,37170.0,2,7,0.66,1
2323,NA,technology,employed,europe,84780.0,2,2,0.33,1
4715,referral,retail,employed,north_america,41804.0,2,4,0.52,0
1018,social_media,technology,unemployed,north_america,0.0,1,0,0.20,0
1666,paid_ads,retail,student,europe,23318.0,0,0,0.04,0


In [104]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [105]:
df[['annual_income','lead_score','number_of_courses_viewed','interaction_count']].corr().round(3)

,annual_income,lead_score,number_of_courses_viewed,interaction_count
annual_income,1.000,0.229,0.161,0.123
lead_score,0.229,1.000,0.757,0.916
number_of_courses_viewed,0.161,0.757,1.000,0.722
interaction_count,0.123,0.916,0.722,1.000


In [106]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [107]:
for c in ['lead_source', 'industry', 'employment_status', 'location']:
    print(f"The column MI for {c} is: {mutual_info_score(df_train.converted, df_train[c]).round(3)}")

The column MI for lead_source is: 0.034
The column MI for industry is: 0.002
The column MI for employment_status is: 0.019
The column MI for location is: 0.001


In [153]:
df1 = df.copy()

In [154]:
df1.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [155]:
df_full_train, df_test = train_test_split(df1, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [156]:
df_test = df_test.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [157]:
y_test = df_test['converted']
y_train = df_train['converted']
y_val = df_val['converted']

In [158]:
del df_test['converted']
del df_train['converted']
del df_val['converted']

#### Q3

In [159]:
from sklearn.feature_extraction import DictVectorizer

In [160]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
dv.fit(train_dicts)
X_transform = dv.transform(train_dicts)

In [161]:
val_dicts = df_val.to_dict(orient='records')
val_transform = dv.transform(val_dicts)

In [148]:
from sklearn.linear_model import LogisticRegression

In [149]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_transform, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [150]:
y_pred = model.predict_proba(val_transform)[:,1]

In [151]:
prediction = (y_pred > 0.5).astype(int)

In [152]:
(y_val == prediction).mean()

np.float64(0.601)

In [162]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_transform, y_train)
    y_pred = model.predict_proba(val_transform)[:,1]
    prediction = (y_pred > 0.5).astype(int)
    print((y_val == prediction).mean())

0.598
0.598
0.613
0.645


In [80]:
def logReg(X_train, X_val, y_train, y_val):
    train_dicts = X_train.to_dict(orient='records')
    dv.fit(train_dicts)
    X_transform = dv.transform(train_dicts)
    
    val_dicts = X_val.to_dict(orient='records')
    val_transform = dv.transform(val_dicts)
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_transform, y_train)
    y_pred = model.predict_proba(val_transform)[:,1]
    price_prediction = (y_pred >= 0.5)
    result = (y_val == price_prediction).mean()
    return(result)

In [82]:
df_train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
0,-120.97,38.00,27.0,1683.0,288.0,873.0,258.0,4.7069,INLAND,6.523256,0.171123,3.383721
1,-117.87,34.10,15.0,6409.0,1363.0,3359.0,1267.0,3.8750,<1H OCEAN,5.058406,0.212670,2.651144
2,-117.85,33.76,26.0,2312.0,525.0,1273.0,437.0,2.8828,<1H OCEAN,5.290618,0.227076,2.913043
3,-119.74,34.35,34.0,1664.0,292.0,705.0,257.0,5.0000,NEAR OCEAN,6.474708,0.175481,2.743191
4,-121.51,38.50,25.0,4719.0,745.0,1857.0,739.0,5.0371,INLAND,6.385656,0.157872,2.512855


In [90]:
df_train.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household'],
      dtype='object')

In [91]:
df_train1 = df_train[['longitude', 'latitude', 'housing_median_age',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household']]

In [94]:
score = logReg(df_train1, df_val, y_train, y_val)
score

0.8292461398728429

In [95]:
df_train2 = df_train[['longitude', 'latitude', 'housing_median_age', 'total_rooms',
      'population', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household']]

In [96]:
score = logReg(df_train2, df_val, y_train, y_val)
score

0.828640629730548

In [97]:
df_train3 = df_train[['longitude', 'latitude', 'housing_median_age','total_rooms',
       'total_bedrooms', 'households', 'median_income',
       'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household']]

In [98]:
score = logReg(df_train3, df_val, y_train, y_val)
score

0.8207689978807146

In [99]:
df_train4 = df_train[['longitude', 'latitude', 'housing_median_age','total_rooms',
       'total_bedrooms', 'median_income','population',
       'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household']]

In [100]:
score = logReg(df_train4, df_val, y_train, y_val)
score

0.823191038449894